# 04 Gold KPIs
Likely out-of-stocks are inferred from runs of zero sales: if a product normally sells v units/day,
k zero days in a row happen by chance with probability ≈ exp(-v·k) (Poisson). Below 1% → likely out of stock.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

s = spark.table("workspace.m5.silver_sales_daily")
w_hist = Window.partitionBy("id").orderBy("date").rowsBetween(-28, -1)
s = s.withColumn("velocity_28d", F.avg("units").over(w_hist))

In [0]:
w_id      = Window.partitionBy("id").orderBy("date")
w_id_zero = Window.partitionBy("id", "is_zero").orderBy("date")

z = (s.withColumn("is_zero", (F.col("units") == 0).cast("int"))
      .withColumn("grp", F.row_number().over(w_id) - F.row_number().over(w_id_zero)))

runs = (z.filter("is_zero = 1")
          .groupBy("id", "item_id", "dept_id", "cat_id", "store_id", "grp")
          .agg(F.min("date").alias("run_start"),
               F.max("date").alias("run_end"),
               F.count("*").alias("zero_days"),
               F.min_by("velocity_28d", "date").alias("velocity_before"),
               F.avg("sell_price").alias("price")))

In [0]:
oos = (runs.filter(F.col("velocity_before") > 0)
           .withColumn("p_by_chance", F.exp(-F.col("velocity_before") * F.col("zero_days")))
           .filter(F.col("p_by_chance") < 0.01)
           .withColumn("lost_units_est",   F.col("velocity_before") * F.col("zero_days"))
           .withColumn("lost_revenue_est", F.col("lost_units_est") * F.coalesce("price", F.lit(0))))

oos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.m5.gold_oos_events")
print(f"likely out-of-stock events: {oos.count():,}")
display(oos.orderBy(F.desc("lost_revenue_est")).limit(20))

In [0]:
oos_days = (spark.table("workspace.m5.gold_oos_events")
    .select("id",
            F.explode(F.sequence("run_start", "run_end")).alias("date"),
            (F.col("lost_units_est") / F.col("zero_days")).alias("lost_units_day"),
            (F.col("lost_revenue_est") / F.col("zero_days")).alias("lost_rev_day"))
    .withColumn("oos", F.lit(1)))

avail = (spark.table("workspace.m5.silver_sales_daily")
    .select("id", "store_id", "dept_id", "date", "units", "revenue")
    .join(oos_days, ["id", "date"], "left")
    .withColumn("week", F.date_trunc("week", "date"))
    .groupBy("store_id", "dept_id", "week")
    .agg(F.count("*").alias("item_days"),
         F.sum(F.coalesce("oos", F.lit(0))).alias("oos_item_days"),
         F.sum("units").alias("units"),
         F.sum("revenue").alias("revenue"),
         F.sum(F.coalesce("lost_units_day", F.lit(0))).alias("lost_units_est"),
         F.sum(F.coalesce("lost_rev_day", F.lit(0))).alias("lost_revenue_est"))
    .withColumn("availability_pct", 1 - F.col("oos_item_days") / F.col("item_days")))

avail.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.m5.gold_store_availability_weekly")
display(avail.groupBy("store_id").agg(F.avg("availability_pct").alias("avg_availability"),
                                      F.sum("lost_revenue_est").alias("lost_revenue_est"))
             .orderBy("avg_availability"))

In [0]:
s2 = spark.table("workspace.m5.silver_sales_daily")
max_date = s2.agg(F.max("date")).first()[0]
rev = (s2.filter(F.col("date") > F.date_sub(F.lit(max_date), 90))
         .groupBy("store_id", "item_id").agg(F.sum("revenue").alias("rev")))

w_rank = Window.partitionBy("store_id").orderBy(F.desc("rev")).rowsBetween(Window.unboundedPreceding, 0)
w_all  = Window.partitionBy("store_id")

abc = (rev.withColumn("cum_share", F.sum("rev").over(w_rank) / F.sum("rev").over(w_all))
          .withColumn("abc", F.when(F.col("cum_share") <= 0.8, "A")
                              .when(F.col("cum_share") <= 0.95, "B").otherwise("C")))
abc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.m5.gold_item_abc")

In [0]:
top = (spark.table("workspace.m5.gold_oos_events")
         .join(spark.table("workspace.m5.gold_item_abc").select("store_id", "item_id", "abc"),
               ["store_id", "item_id"])
         .filter("abc = 'A'")
         .groupBy("store_id", "item_id")
         .agg(F.sum("lost_revenue_est").alias("lost_revenue_est"), F.count("*").alias("oos_events"))
         .orderBy(F.desc("lost_revenue_est")))
top.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.m5.gold_priority_items")
display(top.limit(20))